# UR3_RTDE_Tests — UR3e (PolyScope X) arm bring-up

Arm-only sanity checks for the lab **UR3e on PolyScope X 10.12** (`192.168.1.4`),
using `UR3RealRobotPick` from `ur3_realrobot_dependencies.py` (this folder).

1. Connection test — receive interface + current state (no pendant program needed)
2. Receive test — live stream
3. Control test — `connect_arm()` (External Control URCap) + moveJ to `tucked`

**Arm control on PolyScope X:** the headless script `ur_rtde` would upload does **not**
run on PolyScope X, so the arm **always** uses the **External Control URCap**.
`connect_arm()` opens RTDE receive + control and **blocks until** the pendant's
*External Control* program (Host IP **192.168.1.89** = this PC, port **50002**) is
**PLAYING** with the robot in **Remote Control**.

The Hand-E gripper is a separate channel — see `robots/hande/HandE_Tests.ipynb`.

In [ ]:
# Setup: backend env, imports, robot handle
import os
import platform
import sys
import time

import numpy as np

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=1"
if platform.system() == "Darwin":
    os.environ["MUJOCO_GL"] = "glfw"
    os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
else:
    os.environ.setdefault("MUJOCO_GL", "egl")

import rtde_receive

# self-locate ur3_realrobot_dependencies.py regardless of kernel cwd
# (VSCode runs notebooks from the workspace root).
for _d in (os.getcwd(), os.path.join(os.getcwd(), "robots", "UR3e"),
           os.path.abspath(os.path.join(os.getcwd(), "..", "UR3e"))):
    if os.path.exists(os.path.join(_d, "ur3_realrobot_dependencies.py")):
        sys.path.insert(0, _d)
        break
else:
    raise FileNotFoundError("ur3_realrobot_dependencies.py not found on any candidate path")
from ur3_realrobot_dependencies import UR3RealRobotPick

ROBOT_IP = "192.168.1.4"      # UR3e, PolyScope X 10.12
PC_HOST_IP = "192.168.1.89"   # this PC; the External Control node's Host IP
UR_CAP_PORT = 50002           # External Control URCap port
# 'tucked' keyframe arm pose (6 arm joints) from
# my_ur3/xmls/mjx_single_cube_position_ur3.xml
Q_TUCKED = [0.0, -2.36, 2.36, -1.57, -1.57, 0.0]

# The arm ALWAYS connects via the External Control URCap; connect_arm() (section 3)
# blocks until Play. Sections 1-2 use a raw receive interface so we can verify
# network reachability BEFORE the pendant program is running.
robot = UR3RealRobotPick(host=ROBOT_IP, ur_cap_port=UR_CAP_PORT)
print("ROBOT_IP", ROBOT_IP, "| wrapper handle ready")

## 1. Connection test — receive interface + current state

Opens the raw `rtde_receive` interface (joint feedback only — no control yet) and reads the
current state. This does **not** need the pendant program running; it just confirms the robot
is reachable on the network and streaming RTDE data.

In [ ]:
recv = rtde_receive.RTDEReceiveInterface(ROBOT_IP)
print("receive connected:", recv.isConnected())

print("runtime state (2=PLAYING):", recv.getRuntimeState())
print("robot mode  (7=RUNNING):  ", recv.getRobotMode())
print("q   =", [round(v, 4) for v in recv.getActualQ()])
print("qd  =", [round(v, 4) for v in recv.getActualQd()])
print("tcp =", [round(v, 4) for v in recv.getActualTCPPose()])

## 2. Receive test — live stream

Reads the raw `recv` interface a few times so we can eyeball that joint/TCP/force feedback is
sane and updating.

In [ ]:
for _ in range(5):
    print(
        "q =", [round(v, 4) for v in recv.getActualQ()],
        "| tcp =", [round(v, 4) for v in recv.getActualTCPPose()[:3]],
        "| force =", [round(v, 2) for v in recv.getActualTCPForce()],
    )
    time.sleep(0.2)

## 3. Control test — `connect_arm()` + moveJ to `tucked`

Uses the wrapper's **`connect_arm()`** (External Control URCap, `FLAG_USE_EXT_UR_CAP`,
port 50002). It **blocks until the pendant's External Control program is PLAYING** — so
**press Play now** (External Control node → Host IP `192.168.1.89`, port `50002`; robot in
**Remote Control**). Then `move_to_start()` sends a `moveJ` to the `tucked` pose and polls
until converged.

In [ ]:
# BLOCKS until the pendant External Control program is PLAYING -> press Play first.
robot.connect_arm()
print("arm connected:", robot.is_connected())

# moveJ to the tucked keyframe and poll until converged.
robot.move_to_start(Q_TUCKED, a=0.4, v=0.4, tol=0.01)
robot.print_feedback()

## Teardown

Stops servo motion and closes the RTDE connection (and the gripper handle, if one was
connected on this `robot`).

In [ ]:
robot.disconnect()      # servoStop + stopScript + close RTDE (+ gripper if connected)
try:
    recv.disconnect()
except NameError:
    pass
print("disconnected")